In [0]:
from pyspark.sql.functions import *
from datetime import *
# Get batch_id from widget
dbutils.widgets.text("batch_id", "")
batch_id = dbutils.widgets.get("batch_id")
print(batch_id)

102


In [0]:
# Read metadata for this batch
master_tbl_df = spark.sql(f"SELECT * FROM metadata.master_tble WHERE batch_id = '{batch_id}'")
display(master_tbl_df)

batch_id,source_name,data_load_strategy,source_tbl_name,bronze_tbl_name,silver_tbl_name,watermark_column
102,azure_sql,DeltaLoad_SCD2,dbo.customer,bronze.customer,silver.customer,2000-09-07T15:23:08.535Z


In [0]:
row=master_tbl_df.first()
src_tbl_name_dynamic=row["source_tbl_name"]
bronze_tbl_name_dynamic=row["bronze_tbl_name"]
silver_tbl_name_dynamic=row["silver_tbl_name"]
data_load_strategy=row["data_load_strategy"]
prev_modified_date_dynamic=row["watermark_column"]

print("Source:",src_tbl_name_dynamic)
print("Bronze:",bronze_tbl_name_dynamic)
print("Silver:",silver_tbl_name_dynamic)
print("Strategy:",data_load_strategy)
print("Prev watermark:",prev_modified_date_dynamic)

Source: dbo.customer
Bronze: bronze.customer
Silver: silver.customer
Strategy: DeltaLoad_SCD2
Prev watermark: 2000-09-07 15:23:08.535000


In [0]:
#jdbc connection
src_url=(
    f"jdbc:sqlserver://dm-project1.database.windows.net:1433;"
    f"database=Sales_db;"
    f"user=dmuser;"
    f"password=Dipti123@+;"
)

In [0]:
# Format watermark for SQL Server
try:
    dt = datetime.strptime(str(prev_modified_date_dynamic), '%Y-%m-%d %H:%M:%S.%f')
    formatted_date = dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]  # keep milliseconds
except ValueError:
    raise ValueError("Invalid watermark format")


In [0]:
# Read source incrementally
filter_condition = f"ModifiedDate > '{formatted_date}'"
source_df = (
    spark.read.format("jdbc")
    .option("url", src_url)
    .option("dbtable", f"(SELECT * FROM {src_tbl_name_dynamic} WHERE {filter_condition}) AS temp")
    .load()
)
display(source_df)


customerID,Title,FirstName,MiddleName,LastName,CompanyName,EmailAddress,Phone,ModifiedDate
1,Mr.,John,null,Doe,Contoso Ltd,john.doe@contoso.com,+1-555-1234,2025-11-01T00:00:00.000Z
2,Ms.,Jane,A.,Smith,Fabrikam Inc,jane.smith@fabrikam.com,+1-555-5678,2025-11-02T00:00:00.000Z
3,Dr.,Raj,null,Patel,HealthCare Co,raj.patel@healthcare.org,+91-9876543210,2025-11-03T00:00:00.000Z
5,Mr.,null,null,Lee,null,null,null,2025-11-04T00:00:00.000Z
6,Mrs.,Anna,null,Brown,TechCorp,anna.brown@techcorp.com,+44-20-123456,2025-11-05T00:00:00.000Z
7,Mr.,Carlos,M.,Gomez,RetailCo,carlos.gomez@retailco.com,+34-600-111222,2025-11-06T00:00:00.000Z
8,Ms.,Fatima,null,Khan,EduWorld,fatima.khan@eduworld.org,+971-50-987654,2025-11-07T00:00:00.000Z
9,Dr.,Li,null,Wang,BioLabs,li.wang@biolabs.cn,+86-10-555555,2025-11-08T00:00:00.000Z
10,Mr.,Peter,null,Brien,FinancePlus,peter.obrien@financeplus.com,+353-1-222333,2025-11-09T00:00:00.000Z


In [0]:
# Compute new watermark
src_max_modified_date = source_df.agg(max("ModifiedDate")).first()[0]
print("New watermark:", src_max_modified_date)


New watermark: 2025-11-09 00:00:00


In [0]:
# Apply strategy
if not src_max_modified_date:
    print("No new data to process")

elif data_load_strategy == "DeltaLoad_SCD2":
    # Deduplicate + audit
    source_df_add_col = source_df.dropDuplicates().withColumn("load_date", current_timestamp())

    # Write bronze
    source_df_add_col.write.mode("overwrite").saveAsTable(bronze_tbl_name_dynamic)

    # MERGE into silver (SCD2)
    silver_query = f"""
    MERGE INTO {silver_tbl_name_dynamic} AS target
    USING (
        SELECT DISTINCT
            customerID, Title, FirstName, MiddleName, LastName,
            CompanyName, EmailAddress, Phone, ModifiedDate, load_date,
            current_timestamp() AS start_date,
            TIMESTAMP('9999-12-31 00:00:00') AS end_date,
            'Y' AS is_active
        FROM {bronze_tbl_name_dynamic}
    ) AS source
    ON target.customerID = source.customerID AND target.is_active = 'Y'
       AND (
            target.Title <> source.Title OR
            target.FirstName <> source.FirstName OR
            target.MiddleName <> source.MiddleName OR
            target.LastName <> source.LastName OR
            target.CompanyName <> source.CompanyName OR
            target.EmailAddress <> source.EmailAddress OR
            target.Phone <> source.Phone
       )
    WHEN MATCHED THEN UPDATE SET
        target.end_date = source.start_date,
        target.is_active = 'N'
    WHEN NOT MATCHED THEN INSERT (
        customerID, Title, FirstName, MiddleName, LastName,
        CompanyName, EmailAddress, Phone,
        load_date, start_date, end_date, is_active
    )
    VALUES (
        source.customerID, source.Title, source.FirstName, source.MiddleName, source.LastName,
        source.CompanyName, source.EmailAddress, source.Phone,
        source.load_date, source.start_date, source.end_date, source.is_active
    )
    """
    spark.sql(silver_query)

    # Update watermark
    spark.sql(f"""
        UPDATE metadata.master_tble
        SET watermark_column = '{src_max_modified_date}'
        WHERE batch_id = '{batch_id}'
    """)
else:
    print("Unsupported strategy:", data_load_strategy)


In [0]:
%sql
SELECT * FROM bronze.customer;

customerID,Title,FirstName,MiddleName,LastName,CompanyName,EmailAddress,Phone,ModifiedDate,load_date
9,Dr.,Li,null,Wang,BioLabs,li.wang@biolabs.cn,+86-10-555555,2025-11-08T00:00:00.000Z,2025-11-30T09:45:36.869Z
10,Mr.,Peter,null,Brien,FinancePlus,peter.obrien@financeplus.com,+353-1-222333,2025-11-09T00:00:00.000Z,2025-11-30T09:45:36.869Z
2,Ms.,Jane,A.,Smith,Fabrikam Inc,jane.smith@fabrikam.com,+1-555-5678,2025-11-02T00:00:00.000Z,2025-11-30T09:45:36.869Z
8,Ms.,Fatima,null,Khan,EduWorld,fatima.khan@eduworld.org,+971-50-987654,2025-11-07T00:00:00.000Z,2025-11-30T09:45:36.869Z
3,Dr.,Raj,null,Patel,HealthCare Co,raj.patel@healthcare.org,+91-9876543210,2025-11-03T00:00:00.000Z,2025-11-30T09:45:36.869Z
7,Mr.,Carlos,M.,Gomez,RetailCo,carlos.gomez@retailco.com,+34-600-111222,2025-11-06T00:00:00.000Z,2025-11-30T09:45:36.869Z
1,Mr.,John,null,Doe,Contoso Ltd,john.doe@contoso.com,+1-555-1234,2025-11-01T00:00:00.000Z,2025-11-30T09:45:36.869Z
6,Mrs.,Anna,null,Brown,TechCorp,anna.brown@techcorp.com,+44-20-123456,2025-11-05T00:00:00.000Z,2025-11-30T09:45:36.869Z
5,Mr.,null,null,Lee,null,null,null,2025-11-04T00:00:00.000Z,2025-11-30T09:45:36.869Z


In [0]:
%sql
SELECT * FROM silver.customer;

customerID,Title,FirstName,MiddleName,LastName,CompanyName,EmailAddress,Phone,ModifiedDate,load_date,start_date,end_date,is_active
8,Ms.,Fatima,null,Khan,EduWorld,fatima.khan@eduworld.org,+971-50-987654,null,2025-11-30T09:45:36.869Z,2025-11-30T09:45:38.333Z,9999-12-31T00:00:00.000Z,Y
5,Mr.,null,null,Lee,null,null,null,null,2025-11-30T09:45:36.869Z,2025-11-30T09:45:38.333Z,9999-12-31T00:00:00.000Z,Y
1,Mr.,John,null,Doe,Contoso Ltd,john.doe@contoso.com,+1-555-1234,null,2025-11-30T09:45:36.869Z,2025-11-30T09:45:38.333Z,9999-12-31T00:00:00.000Z,Y
7,Mr.,Carlos,M.,Gomez,RetailCo,carlos.gomez@retailco.com,+34-600-111222,null,2025-11-30T09:45:36.869Z,2025-11-30T09:45:38.333Z,9999-12-31T00:00:00.000Z,Y
10,Mr.,Peter,null,Brien,FinancePlus,peter.obrien@financeplus.com,+353-1-222333,null,2025-11-30T09:45:36.869Z,2025-11-30T09:45:38.333Z,9999-12-31T00:00:00.000Z,Y
2,Ms.,Jane,A.,Smith,Fabrikam Inc,jane.smith@fabrikam.com,+1-555-5678,null,2025-11-30T09:45:36.869Z,2025-11-30T09:45:38.333Z,9999-12-31T00:00:00.000Z,Y
3,Dr.,Raj,null,Patel,HealthCare Co,raj.patel@healthcare.org,+91-9876543210,null,2025-11-30T09:45:36.869Z,2025-11-30T09:45:38.333Z,9999-12-31T00:00:00.000Z,Y
6,Mrs.,Anna,null,Brown,TechCorp,anna.brown@techcorp.com,+44-20-123456,null,2025-11-30T09:45:36.869Z,2025-11-30T09:45:38.333Z,9999-12-31T00:00:00.000Z,Y
9,Dr.,Li,null,Wang,BioLabs,li.wang@biolabs.cn,+86-10-555555,null,2025-11-30T09:45:36.869Z,2025-11-30T09:45:38.333Z,9999-12-31T00:00:00.000Z,Y


In [0]:
%sql
 CREATE OR REPLACE VIEW gold.customer_sales_summary AS
 SELECT
     c.customerID,
     CONCAT(c.FirstName, ' ', c.LastName) AS full_name,
     c.Title,
     c.CompanyName,
     c.EmailAddress,
     c.Phone,
     COALESCE(SUM(s.orderValue), 0) AS total_sales_value,
     COALESCE(MAX(s.orderDate), NULL) AS last_order_date,
     c.is_active
 FROM
     silver.customer c
 LEFT JOIN
     silver.sales s
 ON
     c.customerID = s.customerID
 where s.orderDate is not NULL
 GROUP BY
     c.customerID,
     c.FirstName,
     c.LastName,
     c.Title,
     c.CompanyName,
     c.EmailAddress,
     c.Phone,
     c.is_active;



In [0]:
%sql
SELECT * FROM gold.customer_sales_summary;

customerID,full_name,Title,CompanyName,EmailAddress,Phone,total_sales_value,last_order_date,is_active
6,Anna Brown,Mrs.,TechCorp,anna.brown@techcorp.com,+44-20-123456,2200.00,2025-11-18T00:00:00.000Z,Y
5,null,Mr.,null,null,null,1800.00,2025-11-17T00:00:00.000Z,Y
3,Raj Patel,Dr.,HealthCare Co,raj.patel@healthcare.org,+91-9876543210,3200.75,2025-11-12T00:00:00.000Z,Y
7,Carlos Gomez,Mr.,RetailCo,carlos.gomez@retailco.com,+34-600-111222,3000.00,2025-11-19T00:00:00.000Z,Y
1,John Doe,Mr.,Contoso Ltd,john.doe@contoso.com,+1-555-1234,2500.00,2025-11-10T00:00:00.000Z,Y
2,Jane Smith,Ms.,Fabrikam Inc,jane.smith@fabrikam.com,+1-555-5678,1500.50,2025-11-11T00:00:00.000Z,Y
